# Section 05f Onboarding Notebook: Multireference Electronic Structure — `CASSCF`, `MRCI` (`+Q`), and `MRCC` (`BW-MRCC` & `Mk-MRCC`)

### Overview & Pedagogical Design
In **Section 05 (`05_correlated_wavefunction_methods_notebook.ipynb`)**, we studied single-reference post-Hartree–Fock methods (`MP2`, `CISD`, `CCSD`) built atop a single RHF determinant $|\Phi_0\rangle$. While single-reference `CCSD(T)` is the "gold standard" near equilibrium geometries where $|\Phi_0\rangle$ dominates ($|c_0|^2 > 0.90$, $\mathcal{T}_1 < 0.02$), it breaks down whenever two or more electronic configurations become nearly degenerate — a regime governed by **static (non-dynamical) electron correlation**:
- Homolytic covalent bond breaking (e.g., $\text{H}_2$, $\text{N}_2$, $\text{F}_2$ dissociation)
- Diradicals, transition states, and twisted $\pi$-bonds (e.g., $90^\circ$-twisted ethylene)
- Open-shell transition-metal complexes, lanthanides/actinides, and conical intersections

To treat both **static correlation** (near-degeneracy within a valence active space) and **dynamical correlation** (short-range Coulomb cusps involving core and virtual orbitals) on an equal footing, modern quantum chemistry relies on the **Multireference Hierarchy** ([PySCF MCSCF Guide](https://pyscf.org/user/pbc/scf.html); [ORCA CASSCF/NEVPT2 Manual](https://www.faccts.de/docs/orca/6.0/manual/_downloads/f5a34d500b44971b2b057d96d7f899ca/orca.pdf)):

| Method | Wavefunction Ansatz | Captures Static Correlation? | Captures Dynamical Correlation? | Size-Extensive? |
| :--- | :--- | :--- | :--- | :--- |
| **`CASCI(n, m)`** | $\sum_{\mu \in \mathcal{P}_{\text{CAS}}} c_\mu |\Phi_\mu\rangle$ (fixed MOs) | Yes (within fixed active orbitals) | No | Yes (if active space separates) |
| **`CASSCF(n, m)`** | $\exp(\hat{\kappa})\sum_{\mu \in \mathcal{P}_{\text{CAS}}} c_\mu |\Phi_\mu\rangle$ (variational MOs + CI) | **Yes (optimal active MOs)** | No (active space only) | **Yes** |
| **`MRCISD` & `ic-MRCI`** | $\sum_{\mu \in \mathcal{P}} c_\mu |\Phi_\mu\rangle + \sum_{l \in \mathcal{Q}_{\text{SD}}} c_l |\Phi_l\rangle$ | **Yes** | **Yes** (excitations into virtuals) | No (fixed by **Davidson `+Q`**) |
| **`BW-MRCC` & `Mk-MRCC`** | $\sum_{\mu \in \mathcal{P}} c_\mu e^{\hat{T}(\mu)}|\Phi_\mu\rangle$ (Jeziorski–Monkhorst) | **Yes** | **Yes** (infinite-order cluster) | **Yes (`Mk-MRCC` is strictly size-extensive)** |

In this notebook, **no multireference functions are pre-solved for you**. You will implement **15 granular `# TODO` functions** walking step-by-step through `CASCI`, `CASSCF`, uncontracted & internally contracted `MRCI` with Davidson `+Q` corrections, and state-specific `BW-MRCC` and Mukherjee `Mk-MRCC`:
- **Part 1 — Static Correlation, Active-Space Partitioning $(\mathcal{I}, \mathcal{A}, \mathcal{V})$, Frozen-Core Fock Matrix & `CASCI` (`TODO 1` – `TODO 3`)**
- **Part 2 — Complete Active Space SCF (`CASSCF`): Generalized Fock Matrix, Generalized Brillouin Theorem & Exponential Orbital Rotations $\mathbf{C}e^{\boldsymbol{\kappa}}$ (`TODO 4` – `TODO 7`)**
- **Part 3 — Multireference Configuration Interaction (`MRCISD`, Internal Contraction `ic-MRCI` & Davidson `+Q` Corrections) (`TODO 8` – `TODO 11`)**
- **Part 4 — Multireference Coupled Cluster (`MRCC`): Effective Hamiltonian $\mathbf{H}^{\text{eff}}$, Brillouin–Wigner `BW-MRCC` & Mukherjee `Mk-MRCC` (`TODO 12` – `TODO 15`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section05f_multireference_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh, expm, eig
from pyscf import gto, scf, mcscf, fci
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)


def build_stretched_h4_pyscf_model(stretch_bohr: float = 2.4) -> dict[str, np.ndarray | float | int | object]:
    '''
    Build a rectangular/stretched H4 molecule in the STO-3G basis using PySCF (4 electrons, 4 MOs:
    n_core = 1 doubly occupied core MO, n_act_orb = 2 active frontier MOs, n_virt = 1 external virtual MO).
    As `stretch_bohr` approaches 2.0-2.8 Bohr (near square H4), the frontier HOMO and LUMO become nearly
    degenerate and the system exhibits strong static correlation!
    '''
    mol = gto.M(
        atom=f"H 0.0 0.0 0.0; H 0.0 0.0 1.80; H {stretch_bohr} 0.0 0.0; H {stretch_bohr} 0.0 1.80",
        unit="Bohr",
        basis="sto-3g",
        verbose=0,
    )
    mf = scf.RHF(mol).run(conv_tol=1e-12)
    C_rhf = mf.mo_coeff
    h_ao = mf.get_hcore()
    eri_ao = mol.intor("int2e")
    h_mo = C_rhf.T @ h_ao @ C_rhf
    eri_mo = np.einsum("uvls,up,vq,lr,st->pqrt", eri_ao, C_rhf, C_rhf, C_rhf, C_rhf, optimize=True)

    return {
        "mol": mol,
        "mf": mf,
        "h_ao": h_ao,
        "eri_ao": eri_ao,
        "S_ao": mol.intor("int1e_ovlp"),
        "C_rhf": C_rhf,
        "h_mo": h_mo,
        "eri_mo": eri_mo,
        "e_nuc": float(mol.energy_nuc()),
        "n_elec": 4,
        "n_core": 1,
        "n_act_orb": 2,
        "n_act_elec": 2,
        "n_virt": 1,
    }


print("Section 05f Multireference environment initialized with PySCF. Implement TODOs 1-15 and run `run_section05f_multireference_checkpoints()`!")

## Part 1: Active-Space Partitioning $(\mathcal{I}, \mathcal{A}, \mathcal{V})$, Frozen-Core Fock Matrix & `CASCI` (TODOs 1–3)

### Mathematical Specifications
We partition the $N_{\text{MO}}$ spatial molecular orbitals into three disjoint subspaces:
- **Inactive (Core) Space $\mathcal{I} = \{0, \dots, N_{\text{core}}-1\}$**: Doubly occupied ($n_i = 2$) in all CAS determinants.
- **Active Space $\mathcal{A} = \{N_{\text{core}}, \dots, N_{\text{core}} + M_{\text{act}} - 1\}$**: Contains $N_{\text{act}}$ active electrons correlated via Full CI inside $\mathcal{A}$.
- **External (Virtual) Space $\mathcal{V} = \{N_{\text{core}} + M_{\text{act}}, \dots, N_{\text{MO}} - 1\}$**: Unoccupied ($n_a = 0$) in all CAS determinants.

1. **Inactive Core Fock Matrix & Frozen-Core Energy (`compute_inactive_fock_and_core_energy` — TODO 1)**:
   The $2 N_{\text{core}}$ inactive core electrons create a mean-field Coulomb–Exchange potential that dresses the one-electron integrals of the active orbitals:
   $$F_{pq}^{\text{core}} = h_{pq}^{\text{MO}} + \sum_{i \in \mathcal{I}}\Big[2\,(pq|ii) - (pi|qi)\Big], \qquad E_{\text{core}} = E_{\text{nuc}} + \sum_{i \in \mathcal{I}}\left(h_{ii}^{\text{MO}} + F_{ii}^{\text{core}}\right)$$
   If $N_{\text{core}} == 0$, then $\mathbf{F}^{\text{core}} = \mathbf{h}^{\text{MO}}$ and $E_{\text{core}} = E_{\text{nuc}}$.

2. **Active-Space `CASCI(2e, M_act)` Solver & Correlation Entropy (`solve_active_space_casci_2e` — TODO 2)**:
   Slice the active-space effective one-electron matrix $\mathbf{h}^{\text{act}} = \mathbf{F}^{\text{core}}[\mathcal{A}, \mathcal{A}]$ of shape `(M_act, M_act)` and active two-electron tensor $\mathbf{V}^{\text{act}} = \text{eri\_mo}[\mathcal{A}, \mathcal{A}, \mathcal{A}, \mathcal{A}]$ of shape `(M_act, M_act, M_act, M_act)`. Solve the 2-active-electron singlet Full CI eigenvalue problem inside $\mathcal{A}$:
   $$\mathbf{H}_{\text{CAS}} = \mathbf{h}^{\text{act}} \otimes \mathbf{I} + \mathbf{I} \otimes \mathbf{h}^{\text{act}} + \text{reshape}\big(\text{transpose}(V_{tuvw}^{\text{act}}, (0, 2, 1, 3)),\ (M_{\text{act}}^2, M_{\text{act}}^2)\big)$$
   From the ground-state symmetric spatial wavefunction matrix $C_{tu}$ ($\sum_{tu} |C_{tu}|^2 = 1$):
   - Total `CASCI` energy: $E_{\text{CASCI}} = E_{\text{core}} + E_{\text{act}, 0}$
   - Spin-summed active 1-RDM: $D_{tu}^{\text{act}} = 2\,(\mathbf{C}\mathbf{C}^\mathsf{T})_{tu}$ (with $\text{Tr}\,\mathbf{D}^{\text{act}} = 2$)
   - Spin-summed active 2-RDM: $\Gamma_{tuvw}^{\text{act}} = 2\,C_{tv}\,C_{uw}$ (satisfying $\sum_t D_{tt}^{\text{act}} h_{tt}^{\text{act}} + \frac{1}{2}\sum_{tuvw} \Gamma_{tuvw}^{\text{act}} (tu|vw) = E_{\text{act}, 0}$)
   - Active Natural Orbital Occupation Numbers (NOONs) $n_t \in [0, 2]$ (eigenvalues of $\mathbf{D}^{\text{act}}$ sorted descending) and **von Neumann correlation entropy**:
     $$S_{\text{corr}} = -\sum_{t \in \mathcal{A}} \left[ f_t \ln f_t + (1 - f_t)\ln(1 - f_t) \right], \qquad f_t = \text{clip}\!\left(\frac{n_t}{2},\ 10^{-15},\ 1 - 10^{-15}\right)$$

3. **Diagnosing RHF & MP2 Breakdown Across Bond Stretching (`compare_rhf_mp2_casci_bond_stretch` — TODO 3)**:
   Evaluate the single-determinant energy $E_{\text{SD}}$ (occupying the core MO $0$ and lowest active MO $1$), the `MP2` energy, and the `CASCI(2,2)` energy as the active HOMO–LUMO gap closes!

In [ ]:
def compute_inactive_fock_and_core_energy(
    h_mo: np.ndarray, eri_mo: np.ndarray, e_nuc: float, n_core: int
) -> tuple[np.ndarray, float]:
    '''
    TODO 1: Compute the (n_mo, n_mo) inactive core Fock matrix F_core and scalar frozen-core energy E_core
    for the doubly occupied inactive orbitals 0 .. n_core - 1. Return (F_core, e_core).
    '''
    # TODO 1: If n_core == 0, return (h_mo.copy(), float(e_nuc)).
    #         Otherwise sum (2*(pq|ii) - (pi|qi)) over i in 0..n_core-1 to form F_core,
    #         and compute e_core = e_nuc + sum_{i=0}^{n_core-1} (h_mo[i, i] + F_core[i, i]).
    raise NotImplementedError("Complete TODO 1: compute_inactive_fock_and_core_energy")


def solve_active_space_casci_2e(
    F_core: np.ndarray,
    eri_mo: np.ndarray,
    e_core: float,
    n_core: int,
    n_act_orb: int,
) -> dict[str, float | np.ndarray]:
    '''
    TODO 2: Slice the active space A = slice(n_core, n_core + n_act_orb), solve the 2-active-electron
    singlet Full CI problem on (h_act, V_act), and return a dict with keys:
    'e_casci_tot', 'e_act', 'D_act' (shape M_act x M_act, trace 2),
    'Gamma_act' (shape M_act x M_act x M_act x M_act), 'noons_act', 's_corr_entropy', 'c_mat'.
    '''
    # TODO 2: Slice h_act and V_act, build and diagonalize H_cas of shape (M_act^2, M_act^2),
    #         symmetrize/normalize C_mat, and compute D_act, Gamma_act, NOONs, and s_corr_entropy.
    raise NotImplementedError("Complete TODO 2: solve_active_space_casci_2e")


def compare_rhf_mp2_casci_bond_stretch(
    stretch_grid: np.ndarray,
) -> dict[str, np.ndarray]:
    '''
    TODO 3: For each `s` in `stretch_grid`, build `build_stretched_h4_pyscf_model(s)` and compute:
    1. Single-determinant energy `e_sd` (occupying spatial MOs 0 and 1, i.e., n_core=2 in TODO 1)
    2. MP2 total energy `e_mp2` using canonical diagonal Fock energies eps_p = F_sd[p, p]
    3. CASCI(2e, 2o) total energy `e_casci` (n_core=1, n_act_orb=2 in TODO 1 & TODO 2)
    4. Active LUNO occupation `luno_occ` (second NOON in noons_act) and `s_corr_entropy`.
    Return a dict with keys 'stretch_grid', 'e_sd', 'e_mp2', 'e_casci', 'luno_occ', 's_corr_entropy'.
    '''
    # TODO 3: Loop over stretch_grid, evaluate e_sd, e_mp2, e_casci, luno_occ, and s_corr_entropy.
    raise NotImplementedError("Complete TODO 3: compare_rhf_mp2_casci_bond_stretch")

## Part 2: Complete Active Space SCF (`CASSCF`) — Generalized Fock Matrix, Generalized Brillouin Theorem & Orbital Optimization (TODOs 4–7)

### Mathematical Specifications
1. **Active Fock Matrix $\mathbf{F}^{\text{act}}$ & Generalized Fock Matrix $\mathbf{F}^{\text{gen}}$ (`build_casscf_generalized_fock_and_grad` — TODO 4)**:
   Given the inactive Fock matrix $\mathbf{F}^{\text{core}}$, the active 1-RDM $\mathbf{D}^{\text{act}}$ ($t,u \in \mathcal{A}$), and the active 2-RDM $\boldsymbol{\Gamma}^{\text{act}}$ ($t,u,v,w \in \mathcal{A}$):
   - **Active Fock Matrix** (shape $N_{\text{MO}} \times N_{\text{MO}}$):
     $$F_{pq}^{\text{act}} = \sum_{t,u \in \mathcal{A}} D_{tu}^{\text{act}}\left[ (pq|tu) - \frac{1}{2}(pt|qu) \right]$$
   - **Generalized Fock Matrix** $\mathbf{F}^{\text{gen}}$ (shape $N_{\text{MO}} \times N_{\text{MO}}$, indexed by row subspace):
     $$\begin{aligned}
     F_{iq}^{\text{gen}} &= 2\left(F_{iq}^{\text{core}} + F_{iq}^{\text{act}}\right) \qquad &&(i \in \mathcal{I},\ q \in \mathcal{I}\cup\mathcal{A}\cup\mathcal{V}) \\
     F_{tq}^{\text{gen}} &= \sum_{u \in \mathcal{A}} D_{tu}^{\text{act}}\,F_{qu}^{\text{core}} + \sum_{u,v,w \in \mathcal{A}} \Gamma_{tuvw}^{\text{act}}\,(qu|vw) \qquad &&(t \in \mathcal{A},\ q \in \mathcal{I}\cup\mathcal{A}\cup\mathcal{V}) \\
     F_{aq}^{\text{gen}} &= 0 \qquad &&(a \in \mathcal{V},\ q \in \mathcal{I}\cup\mathcal{A}\cup\mathcal{V})
     \end{aligned}$$
   - **Generalized Brillouin Orbital Gradient**:
     $$G_{pq}^{\text{orb}} = 2\left(F_{pq}^{\text{gen}} - F_{qp}^{\text{gen}}\right)$$
     Zero out the redundant blocks ($\mathcal{I}\text{–}\mathcal{I}$, $\mathcal{A}\text{–}\mathcal{A}$, and $\mathcal{V}\text{–}\mathcal{V}$) so only the **non-redundant inter-subspace blocks** ($\mathcal{I}\text{–}\mathcal{A}$, $\mathcal{I}\text{–}\mathcal{V}$, and $\mathcal{A}\text{–}\mathcal{V}$ and their transposes) remain!

2. **Exponential Unitary Orbital Rotation (`apply_exponential_orbital_rotation` — TODO 5)**:
   Given current MO coefficients $\mathbf{C}$ and an antisymmetric matrix $\boldsymbol{\kappa}$ ($\boldsymbol{\kappa}^\mathsf{T} = -\boldsymbol{\kappa}$), update:
   $$\mathbf{C}_{\text{new}} = \mathbf{C}\,\exp(\boldsymbol{\kappa})$$
   Why use $\exp(\boldsymbol{\kappa})$? Because the matrix exponential of any real skew-symmetric matrix is **strictly orthogonal** ($\mathbf{U}^\mathsf{T}\mathbf{U} = \mathbf{I}$), guaranteeing that $\mathbf{C}_{\text{new}}^\mathsf{T}\mathbf{S}\mathbf{C}_{\text{new}} = \mathbf{I}$ is preserved to machine precision at every macro-iteration!

3. **Quasi-Newton Orbital Rotation Step (`compute_orbital_rotation_step` — TODO 6)** & **Two-Step `CASSCF` Macro-Iteration Solver (`run_two_step_casscf` — TODO 7)**:
   Approximate the diagonal orbital Hessian for non-redundant pairs $(p \in \text{lower subspace},\ q \in \text{higher subspace})$ as $H_{pq}^{\text{diag}} = \max\!\big(2\big|F_{qq}^{\text{core}} + F_{qq}^{\text{act}} - F_{pp}^{\text{core}} - F_{pp}^{\text{act}}\big|,\ h_{\min}\big)$, set $\kappa_{pq} = -G_{pq}^{\text{orb}} / H_{pq}^{\text{diag}}$, and iterate `CASCI` $\leftrightarrow$ $\mathbf{C}\exp(\boldsymbol{\kappa})$ until $\|\mathbf{G}^{\text{orb}}\|_{\text{RMS}} < \text{tol}$!

In [ ]:
def build_casscf_generalized_fock_and_grad(
    F_core: np.ndarray,
    eri_mo: np.ndarray,
    D_act: np.ndarray,
    Gamma_act: np.ndarray,
    n_core: int,
    n_act_orb: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    '''
    TODO 4: Construct the (n_mo, n_mo) active Fock matrix `F_act`, the Generalized Fock matrix `F_gen`,
    the non-redundant antisymmetric orbital rotation gradient matrix `G_orb` (with I-I, A-A, and V-V blocks
    zeroed out), and its RMS norm `rms_grad = sqrt(mean(G_orb^2))`.
    Return (F_act, F_gen, G_orb, rms_grad).
    '''
    # TODO 4: Slice A = slice(n_core, n_core + n_act_orb), form F_act, populate rows of F_gen for I, A, and V,
    #         compute G_orb = 2 * (F_gen - F_gen.T), zero out diagonal subspace blocks, and return (F_act, F_gen, G_orb, rms_grad).
    raise NotImplementedError("Complete TODO 4: build_casscf_generalized_fock_and_grad")


def apply_exponential_orbital_rotation(
    C_mo: np.ndarray, kappa_antisym: np.ndarray
) -> np.ndarray:
    '''
    TODO 5: Enforce skew-symmetry kappa = 0.5 * (kappa_antisym - kappa_antisym.T) and return
    C_new = C_mo @ expm(kappa).
    '''
    # TODO 5: Symmetrize skew-symmetry and apply scipy.linalg.expm.
    raise NotImplementedError("Complete TODO 5: apply_exponential_orbital_rotation")


def compute_orbital_rotation_step(
    G_orb: np.ndarray,
    F_core: np.ndarray,
    F_act: np.ndarray,
    h_min: float = 1.5,
    max_step: float = 0.25,
) -> np.ndarray:
    '''
    TODO 6: Compute the antisymmetric orbital rotation generator `kappa` from `G_orb` using the
    diagonal Fock denominator H_diag[p, q] = max(2 * |F_eff[q, q] - F_eff[p, p]|, h_min), where
    F_eff = F_core + F_act, and clip elements of kappa to [-max_step, +max_step].
    '''
    # TODO 6: Compute f_diag = diag(F_core + F_act), form denom = maximum(2 * |f_diag[None,:] - f_diag[:,None]|, h_min),
    #         set kappa = clip(-G_orb / denom, -max_step, max_step), and return 0.5 * (kappa - kappa.T).
    raise NotImplementedError("Complete TODO 6: compute_orbital_rotation_step")


def run_two_step_casscf(
    h_ao: np.ndarray,
    eri_ao: np.ndarray,
    C_init: np.ndarray,
    e_nuc: float,
    n_core: int,
    n_act_orb: int,
    max_macro: int = 50,
    grad_tol: float = 1e-7,
) -> dict[str, float | int | np.ndarray | list[float]]:
    '''
    TODO 7: Execute the two-step CASSCF macro-iteration loop using TODOs 1, 2, 4, 5, and 6.
    At each macro-iteration:
    1. Transform h_ao and eri_ao into the current MO basis C.
    2. Compute F_core, e_core (TODO 1) and solve CASCI (TODO 2).
    3. Compute F_act, F_gen, G_orb, rms_grad (TODO 4).
    4. If rms_grad < grad_tol, return converged dictionary.
    5. Otherwise compute kappa (TODO 6) and rotate C (TODO 5); if the trial rotation raises the energy,
       backtrack (halve kappa) until E_cas decreases!
    Return a dict with keys:
    'converged', 'e_casscf', 'e_casci_init', 'C_opt', 'D_act', 'Gamma_act',
    'noons_act', 'rms_grad', 'n_macro_iters', 'energy_history'.
    '''
    # TODO 7: Implement the two-step CASSCF macro-iteration solver with backtracking line search.
    raise NotImplementedError("Complete TODO 7: run_two_step_casscf")

## Part 3: Multireference Configuration Interaction (`MRCISD`, Internal Contraction `ic-MRCI` & Davidson `+Q` Corrections) (TODOs 8–11)

### Mathematical Specifications
1. **Generating the Reference ($\mathcal{P}_{\text{CAS}}$) and External (`MRCISD` $\mathcal{Q}_{\text{SD}}$) Determinant Spaces (`generate_mrci_determinant_spaces` — TODO 8)**:
   In the spin-orbital basis ($N_{\text{SO}} = 2 N_{\text{MO}}$):
   - **CAS Reference Space $\mathcal{P}_{\text{CAS}}$**: All $S_z = 0$ Slater determinants (sorted tuples of length $N_{\text{elec}}$) where all $2 N_{\text{core}}$ inactive core spin-orbitals are occupied, $N_{\text{act}}$ active spin-orbitals are occupied (with equal numbers of $\alpha$ and $\beta$ spins), and all $2 N_{\text{virt}}$ virtual spin-orbitals are empty.
   - **External Excitation Space $\mathcal{Q}_{\text{SD}}$**: Every $S_z = 0$ Slater determinant $|D_l\rangle \notin \mathcal{P}_{\text{CAS}}$ that differs by **1 or 2 spin-orbitals** ($1 \le n_{\text{diff}}(D_l, \Phi_\mu) \le 2$) from **at least one** reference determinant $|\Phi_\mu\rangle \in \mathcal{P}_{\text{CAS}}$!

2. **Uncontracted `MRCISD` Hamiltonian Blocks & Solver (`build_and_solve_uncontracted_mrcisd` — TODO 9)**:
   Using the Slater–Condon rules on `all_dets = cas_dets + ext_dets` (of sizes $d_P = |\mathcal{P}_{\text{CAS}}|$ and $d_Q = |\mathcal{Q}_{\text{SD}}|$):
   $$\mathbf{H}_{\text{MRCI}} = \begin{pmatrix} \mathbf{H}_{\mathcal{P}\mathcal{P}} & \mathbf{H}_{\mathcal{P}\mathcal{Q}} \\ \mathbf{H}_{\mathcal{Q}\mathcal{P}} & \mathbf{H}_{\mathcal{Q}\mathcal{Q}} \end{pmatrix}$$
   Diagonalizing $\mathbf{H}_{\text{MRCI}}$ yields the uncontracted $E_{\text{MRCI}} = \lambda_0 + E_{\text{nuc}}$, the dynamical correlation energy $\Delta E_{\text{dyn}}^{\text{MRCI}} = E_{\text{MRCI}} - E_{\text{CAS}}$, and the total CAS reference weight inside the `MRCI` wavefunction:
   $$C_0^2 = \sum_{\mu=0}^{d_P - 1} |c_\mu^{\text{MRCI}}|^2$$

3. **Internally Contracted `ic-MRCI` Projection (`solve_internally_contracted_mrci` — TODO 10)**:
   In **Internally Contracted MRCI (`ic-MRCI`)**, the relative coefficients inside $\mathcal{P}_{\text{CAS}}$ are contracted into the normalized `CASSCF` reference vector $\mathbf{c}_{\text{CAS}}^{(0)}$ (of length $d_P$), defining the $(1 + d_Q)\times(1 + d_Q)$ contracted Hamiltonian:
   $$\mathbf{H}_{\text{ic-MRCI}} = \begin{pmatrix} \mathbf{c}_{\text{CAS}}^{(0)\mathsf{T}}\mathbf{H}_{\mathcal{P}\mathcal{P}}\mathbf{c}_{\text{CAS}}^{(0)} & \mathbf{c}_{\text{CAS}}^{(0)\mathsf{T}}\mathbf{H}_{\mathcal{P}\mathcal{Q}} \\ \mathbf{H}_{\mathcal{Q}\mathcal{P}}\mathbf{c}_{\text{CAS}}^{(0)} & \mathbf{H}_{\mathcal{Q}\mathcal{Q}} \end{pmatrix}$$

4. **Langhoff–Davidson, Renormalized Davidson & Pople `+Q` Size-Extensivity Corrections (`compute_mrci_davidson_q_corrections` — TODO 11)**:
   Because truncated `MRCISD` omits simultaneous quadruple excitations ($\hat{T}_2^2|\Psi_{\text{CAS}}\rangle$), it suffers from size-extensivity errors that grow with the number of correlated electrons $N_{\text{corr}}$. Three standard `+Q` corrections restore approximate size-extensivity:
   $$\begin{aligned}
   \Delta E_{\text{Davidson}} &= \left(1 - C_0^2\right)\left(E_{\text{MRCI}} - E_{\text{CAS}}\right) \\
   \Delta E_{\text{Davidson}}^{\text{renorm}} &= \frac{1 - C_0^2}{C_0^2}\left(E_{\text{MRCI}} - E_{\text{CAS}}\right) \\
   \Delta E_{\text{Pople}} &= \frac{\sqrt{1 + \frac{2(N_{\text{corr}} - 2)}{N_{\text{corr}}}\frac{1 - C_0^2}{C_0^2}} - 1}{\frac{N_{\text{corr}} - 2}{N_{\text{corr}}}}\left(E_{\text{MRCI}} - E_{\text{CAS}}\right)
   \end{aligned}$$

In [ ]:
def generate_mrci_determinant_spaces(
    n_core_spatial: int, n_act_spatial: int, n_virt_spatial: int, n_act_elec: int
) -> tuple[list[tuple[int, ...]], list[tuple[int, ...]]]:
    '''
    TODO 8: Generate the Sz=0 CAS reference determinants `cas_dets` (where all 2*n_core_spatial
    core spin-orbitals are occupied and `n_act_elec` active spin-orbitals are occupied with Sz=0)
    and the external MRCISD determinants `ext_dets` (all Sz=0 determinants outside `cas_dets`
    that differ by 1 or 2 spin-orbitals from at least one determinant in `cas_dets`).
    Return (cas_dets, ext_dets), each sorted lexicographically.
    '''
    # TODO 8: Enumerate all Sz=0 determinants in the full (n_core + n_act + n_virt) spin-orbital space,
    #         partition out cas_dets, and filter ext_dets having min_{mu in cas_dets} n_diff(det, mu) in {1, 2}.
    raise NotImplementedError("Complete TODO 8: generate_mrci_determinant_spaces")


def _slater_condon_so(det_bra: tuple[int, ...], det_ket: tuple[int, ...], h_so: np.ndarray, g_so: np.ndarray) -> float:
    '''Helper: Exact Slater-Condon matrix element <det_bra | H_elec | det_ket> in spin-orbital basis.'''
    set_ket = set(det_ket)
    diff_bra = [p for p in det_bra if p not in set_ket]
    n_diff = len(diff_bra)
    if n_diff >= 3:
        return 0.0
    if n_diff == 0:
        idx = list(det_bra)
        return float(np.sum(h_so[idx, idx]) + 0.5 * np.einsum("pqpq->", g_so[np.ix_(idx, idx, idx, idx)]))
    set_bra = set(det_bra)
    diff_ket = [r for r in det_ket if r not in set_bra]
    sign = (-1) ** (sum(det_bra.index(p) for p in diff_bra) + sum(det_ket.index(r) for r in diff_ket))
    if n_diff == 1:
        p, r = diff_bra[0], diff_ket[0]
        common = [q for q in det_bra if q != p]
        return float(sign * (h_so[p, r] + np.sum(g_so[p, common, r, common])))
    return float(sign * g_so[diff_bra[0], diff_bra[1], diff_ket[0], diff_ket[1]])


def build_and_solve_uncontracted_mrcisd(
    h_so: np.ndarray,
    g_so: np.ndarray,
    e_nuc: float,
    cas_dets: list[tuple[int, ...]],
    ext_dets: list[tuple[int, ...]],
) -> dict[str, float | np.ndarray | int]:
    '''
    TODO 9: Using `_slater_condon_so`, build the full MRCISD Hamiltonian matrix on `cas_dets + ext_dets`,
    extract blocks H_PP, H_PQ, H_QQ, diagonalize H_PP (for E_CAS) and full H_MRCI (for E_MRCI),
    and compute the total CAS reference weight C0_sq = sum_{mu=0}^{d_P-1} |c_mrci[mu]|^2.
    Return a dict with keys:
    'd_P', 'd_Q', 'e_cas_tot', 'e_mrci_tot', 'e_dyn_corr', 'c0_sq',
    'c_cas_vec', 'c_mrci_vec', 'H_PP', 'H_PQ', 'H_QQ'.
    '''
    # TODO 9: Assemble H_mrci, slice H_PP, H_PQ, H_QQ, diagonalize both H_PP and H_mrci with eigh,
    #         and return all 11 items.
    raise NotImplementedError("Complete TODO 9: build_and_solve_uncontracted_mrcisd")


def solve_internally_contracted_mrci(
    H_PP: np.ndarray, H_PQ: np.ndarray, H_QQ: np.ndarray, c_cas_vec: np.ndarray, e_nuc: float
) -> dict[str, float]:
    '''
    TODO 10: Construct the (1 + d_Q, 1 + d_Q) internally contracted MRCI Hamiltonian `H_ic`
    by contracting the P-space with `c_cas_vec`, diagonalize `H_ic` with eigh, and return
    a dict with keys 'e_ic_mrci_tot', 'e_ic_dyn_corr', 'c0_sq_ic', 'contraction_error_eh'
    (where contraction_error_eh is e_ic_mrci_tot - e_uncontracted_mrci_tot).
    '''
    # TODO 10: Build H_ic of shape (1 + d_Q, 1 + d_Q), diagonalize with eigh, and compare with uncontracted H_mrci.
    raise NotImplementedError("Complete TODO 10: solve_internally_contracted_mrci")


def compute_mrci_davidson_q_corrections(
    e_cas_tot: float, e_mrci_tot: float, c0_sq: float, n_elec_corr: int = 4
) -> dict[str, float]:
    '''
    TODO 11: Evaluate the Davidson (+Q_Dav), Renormalized Davidson (+Q_Dav_renorm), and Pople (+Q_Pople)
    size-extensivity corrections and their corresponding total energies:
    Return a dict with keys:
    'de_corr_mrci', 'q_davidson', 'q_davidson_renorm', 'q_pople',
    'e_mrci_q_dav', 'e_mrci_q_renorm', 'e_mrci_q_pople'.
    '''
    # TODO 11: Evaluate all 3 quadruple corrections and add them to e_mrci_tot.
    raise NotImplementedError("Complete TODO 11: compute_mrci_davidson_q_corrections")

## Part 4: Multireference Coupled Cluster (`MRCC`) — Effective Hamiltonian $\mathbf{H}^{\text{eff}}$, Brillouin–Wigner (`BW-MRCC`) & Mukherjee (`Mk-MRCC`) (TODOs 12–15)

### Mathematical Specifications
In Hilbert-space state-specific **Multireference Coupled Cluster (`MRCC`)**, the Jeziorski–Monkhorst wave operator $\hat{\Omega} = \sum_{\mu \in \mathcal{P}} e^{\hat{T}(\mu)} |\Phi_\mu\rangle\langle\Phi_\mu|$ assigns a distinct cluster amplitude vector $T_{l, \mu}$ ($l \in \mathcal{Q}$, $\mu \in \mathcal{P}$) to each active-space reference $|\Phi_\mu\rangle$:

1. **Bloch Effective Hamiltonian $\mathbf{H}^{\text{eff}}$ & Target State Eigenpair (`solve_mrcc_effective_hamiltonian` — TODO 12)**:
   Given the $(d_P \times d_P)$ model-space block $\mathbf{H}_{\mathcal{P}\mathcal{P}}$, the $(d_P \times d_Q)$ coupling block $\mathbf{H}_{\mathcal{P}\mathcal{Q}}$, and the $(d_Q \times d_P)$ cluster amplitude matrix $\mathbf{T}$:
   $$\mathbf{H}^{\text{eff}}(\mathbf{T}) = \mathbf{H}_{\mathcal{P}\mathcal{P}} + \mathbf{H}_{\mathcal{P}\mathcal{Q}}\,\mathbf{T}$$
   Find the lowest real eigenvalue $\mathcal{E}_{\text{elec}}$ and right eigenvector $\mathbf{c}$ of $\mathbf{H}^{\text{eff}}$ (normalized so $\sum_\mu c_\mu^2 = 1$, with signs chosen so $|c_\mu|$ is bounded away from zero via a small regularization `reg_c = 1e-4` when dividing by $c_\mu$).

2. **Brillouin–Wigner MRCC (`BW-MRCC`) Amplitude Step (`step_bw_mrcc_amplitudes` — TODO 13)**:
   Separate $\mathbf{H}_{\mathcal{Q}\mathcal{Q}} = \text{diag}(\mathbf{D}_{\mathcal{Q}}) + \mathbf{V}_{\mathcal{Q}\mathcal{Q}}^{\text{off}}$, where $D_l = (\mathbf{H}_{\mathcal{Q}\mathcal{Q}})_{ll}$ and $\mathbf{V}_{\mathcal{Q}\mathcal{Q}}^{\text{off}}$ has zeros on its diagonal. In `BW-MRCC`, the amplitude equation for reference $\mu \in \mathcal{P}$ and external determinant $l \in \mathcal{Q}$ is:
   $$\left(\mathcal{E}_{\text{elec}} - D_l\right) T_{l, \mu}^{\text{(new)}} = (\mathbf{H}_{\mathcal{Q}\mathcal{P}})_{l\mu} + \sum_{l' \ne l} (\mathbf{V}_{\mathcal{Q}\mathcal{Q}}^{\text{off}})_{ll'}\,T_{l', \mu} - \sum_{\nu \in \mathcal{P}} T_{l, \mu}\,(\mathbf{H}_{\mathcal{P}\mathcal{Q}})_{\mu l'}\,T_{l', \mu}$$
   *(where the quadratic connected/disconnected fold-in term is $-\mathbf{T} \odot (\text{diag}(\mathbf{H}_{\mathcal{P}\mathcal{Q}}\mathbf{T}))$ for each reference column $\mu$).*

3. **Mukherjee State-Specific MRCC (`Mk-MRCC`) Amplitude Step (`step_mk_mrcc_amplitudes` — TODO 14)**:
   To make `MRCC` **strictly size-extensive**, Mahapatra, Datta, and Mukherjee replaced the global eigenvalue $\mathcal{E}_{\text{elec}}$ in the denominator with the reference diagonal $H_{\mu\mu}^{\text{eff}} - D_l$ and added the **Mukherjee inter-reference coupling term**:
   $$\left(D_l - H_{\mu\mu}^{\text{eff}}\right) T_{l, \mu}^{\text{(new)}} = (\mathbf{H}_{\mathcal{Q}\mathcal{P}})_{l\mu} + \left(\mathbf{V}_{\mathcal{Q}\mathcal{Q}}^{\text{off}}\,\mathbf{T}\right)_{l\mu} - T_{l,\mu}\left(\mathbf{H}_{\mathcal{P}\mathcal{Q}}\,\mathbf{T}\right)_{\mu\mu} + \sum_{\nu \ne \mu} \left(T_{l, \nu} - T_{l, \mu}\right) H_{\mu\nu}^{\text{eff}}\,\frac{c_\nu}{c_\mu^{\text{reg}}}$$
   Notice that when all references have identical amplitudes ($T_{l,\nu} = T_{l,\mu}$), the Mukherjee coupling vanishes; when references differ, it couples their cluster operators to guarantee exact separability!

4. **Iterative `BW-MRCC` & `Mk-MRCC` Driver (`run_state_specific_mrcc` — TODO 15)**:
   Iterate `solve_mrcc_effective_hamiltonian` $\leftrightarrow$ amplitude update (with damping `mix = 0.5`) until $|\Delta\mathcal{E}| < \text{tol}$ and $\|\Delta\mathbf{T}\|_{\text{RMS}} < \text{tol}$!

In [ ]:
def solve_mrcc_effective_hamiltonian(
    H_PP: np.ndarray, H_PQ: np.ndarray, T_mat: np.ndarray, e_nuc: float
) -> tuple[np.ndarray, float, float, np.ndarray]:
    '''
    TODO 12: Construct the (d_P, d_P) effective Hamiltonian H_eff = H_PP + H_PQ @ T_mat,
    find its lowest real eigenvalue `e_elec` and corresponding normalized real right eigenvector `c_vec`,
    and return (H_eff, e_elec, e_elec + e_nuc, c_vec).
    '''
    # TODO 12: Form H_eff, compute eigenvalues/eigenvectors via scipy.linalg.eig(H_eff),
    #          pick the index with minimum real eigenvalue, normalize c_vec.real, and return.
    raise NotImplementedError("Complete TODO 12: solve_mrcc_effective_hamiltonian")


def step_bw_mrcc_amplitudes(
    e_elec: float,
    H_PP: np.ndarray,
    H_PQ: np.ndarray,
    H_QQ: np.ndarray,
    T_curr: np.ndarray,
    damping: float = 0.6,
) -> tuple[np.ndarray, float]:
    '''
    TODO 13: Perform one damped Brillouin-Wigner MRCC (BW-MRCC) amplitude update:
    rhs_{l, mu} = H_QP[l, mu] + (V_QQ_off @ T_curr)[l, mu] - T_curr[l, mu] * (H_PQ @ T_curr)[mu, mu]
    T_raw[l, mu] = rhs_{l, mu} / (e_elec - H_QQ[l, l])
    T_new = damping * T_raw + (1 - damping) * T_curr
    Return (T_new, rms_residual) where rms_residual = sqrt(mean((T_raw - T_curr)^2)).
    '''
    # TODO 13: Extract D_Q = diag(H_QQ) and V_QQ_off = H_QQ - diag(D_Q), evaluate rhs, T_raw, T_new, and rms_residual.
    raise NotImplementedError("Complete TODO 13: step_bw_mrcc_amplitudes")


def step_mk_mrcc_amplitudes(
    H_eff: np.ndarray,
    c_vec: np.ndarray,
    H_PP: np.ndarray,
    H_PQ: np.ndarray,
    H_QQ: np.ndarray,
    T_curr: np.ndarray,
    damping: float = 0.5,
    reg_c: float = 1e-3,
) -> tuple[np.ndarray, float]:
    '''
    TODO 14: Perform one damped Mukherjee State-Specific MRCC (Mk-MRCC) amplitude update:
    For each reference mu in 0..d_P-1:
      c_mu_reg = sign(c_vec[mu]) * max(|c_vec[mu]|, reg_c)
      mukherjee_coupling[:, mu] = sum_{nu != mu} (T_curr[:, nu] - T_curr[:, mu]) * H_eff[mu, nu] * (c_vec[nu] / c_mu_reg)
      rhs[:, mu] = H_QP[:, mu] + (V_QQ_off @ T_curr)[:, mu] - T_curr[:, mu] * (H_PQ @ T_curr)[mu, mu] + mukherjee_coupling[:, mu]
      T_raw[:, mu] = rhs[:, mu] / (H_eff[mu, mu] - D_Q)
    Return (T_new, rms_residual) with T_new = damping * T_raw + (1 - damping) * T_curr.
    '''
    # TODO 14: Compute mukherjee_coupling, rhs, T_raw, T_new, and rms_residual.
    raise NotImplementedError("Complete TODO 14: step_mk_mrcc_amplitudes")


def run_state_specific_mrcc(
    H_PP: np.ndarray,
    H_PQ: np.ndarray,
    H_QQ: np.ndarray,
    e_nuc: float,
    method: str = "mk",
    max_iter: int = 120,
    tol: float = 1e-9,
) -> dict[str, float | int | bool | np.ndarray | list[float]]:
    '''
    TODO 15: Run the iterative state-specific MRCC solver (`method="bw"` for BW-MRCC or `method="mk"` for Mk-MRCC)
    starting from T_0 = 0 using TODOs 12, 13, and 14.
    Return a dict with keys:
    'converged', 'method', 'e_mrcc_tot', 'e_dyn_corr', 'T_opt', 'c_vec', 'H_eff', 'n_iterations', 'energy_history'.
    '''
    # TODO 15: Initialize T = zeros((d_Q, d_P)), loop up to max_iter calling solve_mrcc_effective_hamiltonian
    #          and either step_bw_mrcc_amplitudes or step_mk_mrcc_amplitudes until |dE| < tol and rms_res < tol.
    raise NotImplementedError("Complete TODO 15: run_state_specific_mrcc")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section05f_multireference_checkpoints()` below as you implement each `# TODO` function to test your active-space integrals, `CASCI` solver, Generalized Fock matrix, exponential orbital rotations, two-step `CASSCF` macro-optimizer, uncontracted and internally contracted `MRCISD`, Davidson/Pople `+Q` corrections, and state-specific `BW-MRCC` and Mukherjee `Mk-MRCC` solvers!

In [2]:
def _spatial_to_spin_orbital(h_mo: np.ndarray, eri_mo: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    '''Convert spatial MO integrals (h_mo, eri_mo) into spin-orbital h_so and antisymmetrized <pq||rs>.'''
    n_mo = h_mo.shape[0]
    n_so = 2 * n_mo
    h_so = np.zeros((n_so, n_so))
    for p in range(n_mo):
        for q in range(n_mo):
            h_so[2 * p, 2 * q] = h_so[2 * p + 1, 2 * q + 1] = h_mo[p, q]
    eri_phys = np.zeros((n_so, n_so, n_so, n_so))
    for sp in (0, 1):
        for sq in (0, 1):
            eri_phys[sp::2, sq::2, sp::2, sq::2] = eri_mo.transpose(0, 2, 1, 3)
    return h_so, eri_phys - eri_phys.transpose(0, 1, 3, 2)


def run_section05f_multireference_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 15 #TODO functions in Section 05f against PySCF CASCI/CASSCF/FCI.'''
    passed = 0
    total_todos = 15

    try:
        # --- PART 1 CHECKS: ACTIVE SPACE & CASCI (TODOs 1-3) ---
        m_h4 = build_stretched_h4_pyscf_model(stretch_bohr=2.0)
        mc_casci_ref = mcscf.CASCI(m_h4["mf"], ncas=2, nelecas=2).run()
        mc_casscf_ref = mcscf.CASSCF(m_h4["mf"], ncas=2, nelecas=2).run()
        e_fci_ref = fci.FCI(m_h4["mol"], m_h4["mf"].mo_coeff).kernel()[0]

        F_core, e_core = compute_inactive_fock_and_core_energy(
            m_h4["h_mo"], m_h4["eri_mo"], float(m_h4["e_nuc"]), n_core=1
        )
        assert F_core.shape == (4, 4) and np.allclose(F_core, F_core.T)
        passed += 1
        print(f"[PASS] TODO 1 : compute_inactive_fock_and_core_energy (E_core = {e_core:.6f} Eh)")

        casci_res = solve_active_space_casci_2e(F_core, m_h4["eri_mo"], e_core, n_core=1, n_act_orb=2)
        assert np.isclose(casci_res["e_casci_tot"], mc_casci_ref.e_tot, atol=1e-7)
        assert np.isclose(np.trace(casci_res["D_act"]), 2.0, atol=1e-12)
        passed += 1
        print(f"[PASS] TODO 2 : solve_active_space_casci_2e (E_CASCI = {casci_res['e_casci_tot']:.8f} Eh matches PySCF {mc_casci_ref.e_tot:.8f} Eh)")

        scan_s = compare_rhf_mp2_casci_bond_stretch(np.array([1.5, 1.8, 2.0]))
        assert scan_s["e_casci"][-1] < scan_s["e_sd"][-1]
        passed += 1
        print(f"[PASS] TODO 3 : compare_rhf_mp2_casci_bond_stretch (LUNO occ = {scan_s['luno_occ'][-1]:.3f})")

        # --- PART 2 CHECKS: CASSCF ORBITAL OPTIMIZATION (TODOs 4-7) ---
        F_act, F_gen, G_orb, rms_g0 = build_casscf_generalized_fock_and_grad(
            F_core, m_h4["eri_mo"], casci_res["D_act"], casci_res["Gamma_act"], n_core=1, n_act_orb=2
        )
        assert G_orb.shape == (4, 4) and np.allclose(G_orb + G_orb.T, 0.0)
        assert np.linalg.norm(G_orb[1:3, 1:3]) < 1e-14
        passed += 1
        print(f"[PASS] TODO 4 : build_casscf_generalized_fock_and_grad (Initial ||G_orb||_RMS = {rms_g0:.4e})")

        kap_test = np.zeros((4, 4)); kap_test[0, 1] = 0.1; kap_test[1, 0] = -0.1
        C_rot = apply_exponential_orbital_rotation(np.eye(4), kap_test)
        assert np.linalg.norm(C_rot.T @ C_rot - np.eye(4)) < 1e-14
        passed += 1
        print("[PASS] TODO 5 : apply_exponential_orbital_rotation")

        kap_step = compute_orbital_rotation_step(G_orb, F_core, F_act)
        assert np.allclose(kap_step + kap_step.T, 0.0) and np.max(np.abs(kap_step)) <= 0.25
        passed += 1
        print("[PASS] TODO 6 : compute_orbital_rotation_step")

        casscf_res = run_two_step_casscf(
            m_h4["h_mo"], m_h4["eri_mo"], np.eye(4), float(m_h4["e_nuc"]), n_core=1, n_act_orb=2, max_macro=60, grad_tol=1e-6
        )
        assert np.isclose(casscf_res["e_casscf"], mc_casscf_ref.e_tot, atol=1e-5)
        passed += 1
        print(f"[PASS] TODO 7 : run_two_step_casscf (E_CASSCF = {casscf_res['e_casscf']:.8f} Eh matches PySCF {mc_casscf_ref.e_tot:.8f} Eh)")

        # --- PART 3 CHECKS: MRCISD, ic-MRCI & DAVIDSON +Q (TODOs 8-11) ---
        cas_dets, ext_dets = generate_mrci_determinant_spaces(n_core_spatial=1, n_act_spatial=2, n_virt_spatial=1, n_act_elec=2)
        assert len(cas_dets) == 4 and len(ext_dets) == 32
        passed += 1
        print(f"[PASS] TODO 8 : generate_mrci_determinant_spaces (d_P = {len(cas_dets)} CAS dets, d_Q = {len(ext_dets)} external dets)")

        C_opt = casscf_res["C_opt"]
        h_mo_opt = C_opt.T @ m_h4["h_mo"] @ C_opt
        eri_mo_opt = np.einsum("uvls,up,vq,lr,st->pqrt", m_h4["eri_mo"], C_opt, C_opt, C_opt, C_opt, optimize=True)
        h_so, g_so = _spatial_to_spin_orbital(h_mo_opt, eri_mo_opt)

        mrci_res = build_and_solve_uncontracted_mrcisd(h_so, g_so, float(m_h4["e_nuc"]), cas_dets, ext_dets)
        assert np.isclose(mrci_res["e_cas_tot"], casscf_res["e_casscf"], atol=1e-8)
        assert np.isclose(mrci_res["e_mrci_tot"], e_fci_ref, atol=1e-7)
        passed += 1
        print(f"[PASS] TODO 9 : build_and_solve_uncontracted_mrcisd (E_MRCI = {mrci_res['e_mrci_tot']:.8f} Eh matches PySCF FCI {e_fci_ref:.8f} Eh)")

        ic_res = solve_internally_contracted_mrci(
            mrci_res["H_PP"], mrci_res["H_PQ"], mrci_res["H_QQ"], mrci_res["c_cas_vec"], float(m_h4["e_nuc"])
        )
        assert ic_res["e_ic_mrci_tot"] >= mrci_res["e_mrci_tot"] - 1e-12
        passed += 1
        print(f"[PASS] TODO 10: solve_internally_contracted_mrci (E_ic-MRCI = {ic_res['e_ic_mrci_tot']:.8f} Eh)")

        q_res = compute_mrci_davidson_q_corrections(mrci_res["e_cas_tot"], mrci_res["e_mrci_tot"], mrci_res["c0_sq"], n_elec_corr=4)
        assert q_res["q_davidson"] < 0.0 and q_res["q_davidson_renorm"] < q_res["q_davidson"]
        passed += 1
        print(f"[PASS] TODO 11: compute_mrci_davidson_q_corrections (E_MRCI+Q_Dav = {q_res['e_mrci_q_dav']:.8f} Eh)")

        # --- PART 4 CHECKS: BW-MRCC & MUKHERJEE Mk-MRCC (TODOs 12-15) ---
        d_P, d_Q = mrci_res["d_P"], mrci_res["d_Q"]
        H_eff_0, e_el_0, e_tot_0, c_0 = solve_mrcc_effective_hamiltonian(
            mrci_res["H_PP"], mrci_res["H_PQ"], np.zeros((d_Q, d_P)), float(m_h4["e_nuc"])
        )
        assert np.isclose(e_tot_0, mrci_res["e_cas_tot"], atol=1e-10)
        passed += 1
        print("[PASS] TODO 12: solve_mrcc_effective_hamiltonian")

        T_bw_1, res_bw_1 = step_bw_mrcc_amplitudes(e_el_0, mrci_res["H_PP"], mrci_res["H_PQ"], mrci_res["H_QQ"], np.zeros((d_Q, d_P)))
        assert T_bw_1.shape == (d_Q, d_P) and res_bw_1 > 0.0
        passed += 1
        print("[PASS] TODO 13: step_bw_mrcc_amplitudes")

        T_mk_1, res_mk_1 = step_mk_mrcc_amplitudes(H_eff_0, c_0, mrci_res["H_PP"], mrci_res["H_PQ"], mrci_res["H_QQ"], np.zeros((d_Q, d_P)))
        assert T_mk_1.shape == (d_Q, d_P) and res_mk_1 > 0.0
        passed += 1
        print("[PASS] TODO 14: step_mk_mrcc_amplitudes")

        bw_res = run_state_specific_mrcc(mrci_res["H_PP"], mrci_res["H_PQ"], mrci_res["H_QQ"], float(m_h4["e_nuc"]), method="bw")
        mk_res = run_state_specific_mrcc(mrci_res["H_PP"], mrci_res["H_PQ"], mrci_res["H_QQ"], float(m_h4["e_nuc"]), method="mk")
        assert bw_res["converged"] and mk_res["converged"]
        passed += 1
        print(f"[PASS] TODO 15: run_state_specific_mrcc (E_BW-MRCC = {bw_res['e_mrcc_tot']:.8f} Eh, E_Mk-MRCC = {mk_res['e_mrcc_tot']:.8f} Eh)")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 05f (Multireference CASSCF, MRCI & MRCC) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section05f_multireference_checkpoints()
print("Section 05f Multireference checkpoint suite loaded. Implement TODOs 1-15 and call `run_section05f_multireference_checkpoints()`!")

Section 05f Multireference checkpoint suite loaded. Implement TODOs 1-15 and call `run_section05f_multireference_checkpoints()`!
